In [ ]:
import rasterio
import numpy as np
from scipy.ndimage import convolve

# Direct mapping of edge counts based on the encoded values from 0 to 15
def assign_edge_lengths_directly(data):
    # Create an output array initialized to zero
    edge_lengths = np.zeros_like(data, dtype=np.uint8)
    
    # Map from encoded value to number of edges
    edge_count_map = {
        0: 0,  # No edges
        1: 1, 2: 1, 4: 1, 8: 1,  # Single edges
        3: 2, 5: 2, 9: 2, 6: 2, 10: 2, 12: 2,  # Two edges
        7: 3, 11: 3, 13: 3, 14: 3,  # Three edges
        15: 4  # All four edges
    }
    
    # Assign edge lengths based on the edge count map
    for key, value in edge_count_map.items():
        edge_lengths[data == key] = value

    return edge_lengths

# Load the edge map
with rasterio.open("G:/Hangkai/CONUS Forest Edge Mapping/CONUS Forest Edge/2019LC_edges.tif") as src:
    edge_data = src.read(1)

print(1)
# Calculate edge lengths directly in the array
edge_lengths = assign_edge_lengths_directly(edge_data)
del edge_data
print(2)

# Load the forest depth map
with rasterio.open("G:/Hangkai/CONUS Forest Edge Mapping/Forest_Depth_Classification/2019LC.tif") as src:
    depth_data = src.read(1)
print(3)


In [ ]:
# Analyze impact on different forest depths
# Filter to include only depth values between 1 and 3
# Sum the edge lengths within a 5x5 window using the convolution kernel
kernel = np.ones((5, 5), dtype=np.uint8)

# Convolve edge lengths to sum up within the 3x3 window
summed_edge_lengths_direct = convolve(edge_lengths, kernel, mode='constant', cval=0)

with rasterio.open("G:/Hangkai/CONUS Forest Edge Mapping/Forest_Depth_Classification/2019LC.tif") as src:
    existing_crs = src.crs
    transform = src.transform  # Also capture the transform if needed
    
# Prepare metadata for the raster file
metadata = {
    'driver': 'GTiff',
    'dtype': 'uint8',
    'nodata': None,
    'width': summed_edge_lengths_direct.shape[1],
    'height': summed_edge_lengths_direct.shape[0],
    'count': 1,
    'crs': existing_crs,
    'transform': transform
}

# Write the data to a TIFF file
file_path = 'G:/Hangkai/CONUS Forest Edge Mapping/summed_edge_lengths.tif'
with rasterio.open(file_path, 'w', **metadata) as dst:
    dst.write(summed_edge_lengths_direct, 1)



In [ ]:
# Filter depth impacts using the new summed edge lengths and previously defined conditions (depths 1 to 3)
new_depth_impact = {}
for level in depth_levels:
    if 1 <= level <= 3:
        mask = depth_data == level
        total_edge_impact = new_summed_edge_lengths[mask].sum()
        mean_edge_impact = new_summed_edge_lengths[mask].mean() if mask.sum() > 0 else 0
        new_depth_impact[level] = {'total_edge_impact': total_edge_impact, 'average_edge_impact': mean_edge_impact}
    else:
        # Assign 0 or NaN for depths outside the range 1 to 3
        new_depth_impact[level] = {'total_edge_impact': 0, 'average_edge_impact': np.nan}